# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

# EQUIPE



- Leonardo Jae Yong Noh - 11202230594
- Thalita Kelly Ferreira da Costa - 11202131854
- Victor Martim Nascimento - 11202230763

# API

API: THEMOVIEDB

Site oficial:  https://www.themoviedb.org/?language=pt-BR

Link para documentação oficial: https://developer.themoviedb.org/docs/getting-started


# GITHUB

https://github.com/LeonardoJYNoh/NLP-TheMovieDatabaseTMDB

# DESCRIÇÃO

**Reconhecimento de Entidades Nomeadas (NER)**

- Identificar atores e diretores citados em um texto livre e buscar na API os filmes de cada um.
- Exemplo: "Vi um filme do Denis Villeneuve com a Zendaya" → filmografia de cada um.

**Sistemas de Perguntas e Respostas (RAG)**

- Responder perguntas sobre um filme usando apenas os dados que a API retornou como contexto.
- Exemplo: "Quem dirigiu o filme? Quanto tempo ele dura?"

**Classificação de textos**

- Classificar um pedido livre do usuário em gêneros e recomendar filmes reais da API.
- Exemplo: "Quero algo tenso e sombrio para ver de madrugada" → Terror, Suspense.

**Análise de sentimentos, Tradução e Sumarização**

- Ler avaliações em inglês, classificar o sentimento, traduzir e resumir em português e gerar um veredito geral do filme.

# DOCUMENTAÇÃO

- TMDB API — documentação
- TMDB — autenticação
- TMDB — filmes populares
- TMDB — avaliações de filmes
- Google Gemini API
- Google Gemini — modelos
- LangChain — integração Google GenAI
- LangChain — RAG

# BIBLIOTECA / VALIDAÇÕES

### 1. Instalação das bibliotecas

In [1]:
# Instala o que o Colab não traz por padrão (leva ~1 minuto)
!pip install -q langchain langchain-google-genai google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


### 2. Imports

In [2]:
import pandas as pd
import requests

from google import genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

### 3. Chaves de API

In [3]:
# Lê as chaves guardadas em "Secrets" do Colab (ícone de chave, na barra lateral).
# Se não existirem, pede para digitar na hora.
from getpass import getpass

def pegar_chave(nome):
    try:
        return userdata.get(nome)
    except Exception:
        return getpass(f"Digite a chave {nome}: ")

GEMINI_API_KEY = pegar_chave("GEMINI")
TMDB_API_KEY = pegar_chave("TMDB")   # use o "Token de Acesso de Leitura da API" do TMDB

Digite a chave GEMINI: ··········
Digite a chave TMDB: ··········


### 4. LLM (Gemini) e API (TMDB)

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3
)

#versao do llm


In [5]:
cliente = genai.Client(api_key=GEMINI_API_KEY)

modelos = list(cliente.models.list())

print(f"Total geral de modelos: {len(modelos)}")

Total geral de modelos: 28


In [6]:
cliente = genai.Client( api_key= GEMINI_API_KEY)

model_info = cliente.models.get(model="gemini-3.5-flash-lite")
model_info

Model(
  description='Gemini 3.5 Flash Lite',
  display_name='Gemini 3.5 Flash Lite',
  input_token_limit=1048576,
  max_temperature=2.0,
  name='models/gemini-3.5-flash-lite',
  output_token_limit=65536,
  supported_actions=[
    'generateContent',
    'countTokens',
    'createCachedContent',
    'batchGenerateContent',
  ],
  temperature=1.0,
  thinking=True,
  top_k=64,
  top_p=0.95,
  tuned_model_info=TunedModelInfo(),
  version='3.5-flash-lite-07-2026'
)

In [7]:
##autenticação da api

TMDB_URL = "https://api.themoviedb.org/3"
headers = {
    "Authorization": f"Bearer {TMDB_API_KEY}",
    "accept": "application/json"
}

print("Configuração do TMDB concluída!")


Configuração do TMDB concluída!


# COMO USAMOS O LANGCHAIN

Em todas as técnicas o LLM e a API do TMDB trabalham juntos:

> texto do usuário → **LLM** interpreta (técnica de PLN) → **TMDB** busca dados reais → **LLM** responde em português

O LangChain monta cada etapa como uma *chain*: `prompt | llm` (ou `prompt | llm.with_structured_output(Esquema)` quando queremos a resposta em um formato fixo).

In [8]:
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

pd.set_option("display.max_colwidth", 200)

# FUNÇÕES DA API (TMDB)

Uma função simples para consultar o TMDB. Todas as técnicas usam ela.

In [9]:
def tmdb(caminho, **parametros):
    """Consulta o TMDB e devolve o resultado (JSON). Por padrão, em português."""
    parametros.setdefault("language", "pt-BR")
    resp = requests.get(f"{TMDB_URL}{caminho}", headers=headers, params=parametros)
    resp.raise_for_status()
    return resp.json()

# Gêneros oficiais do TMDB: {"ação": 28, "comédia": 35, ...}
mapa_generos = {g["name"].lower(): g["id"] for g in tmdb("/genre/movie/list")["genres"]}
print(list(mapa_generos))

# Filme usado nas técnicas 2 e 4 (pode trocar por qualquer outro)
FILME = "Oppenheimer"

['ação', 'aventura', 'animação', 'comédia', 'crime', 'documentário', 'drama', 'família', 'fantasia', 'história', 'terror', 'música', 'mistério', 'romance', 'ficção científica', 'cinema tv', 'thriller', 'guerra', 'faroeste']


# 1. RECONHECIMENTO DE ENTIDADES NOMEADAS (NER)

O usuário escreve um texto livre. O LLM **identifica os atores e diretores citados** (NER) e o TMDB devolve **os filmes de cada um**.

`texto livre → LLM (NER) → TMDB (busca a pessoa e seus créditos) → filmografia`

In [10]:
class Pessoa(BaseModel):
    nome: str = Field(description="Nome da pessoa, como aparece no texto")
    tipo: Literal["ator", "diretor"] = Field(description="Atores e atrizes são 'ator'; diretores e diretoras são 'diretor'")

class Pessoas(BaseModel):
    pessoas: List[Pessoa] = Field(description="Atores e diretores citados (lista vazia se não houver)")

prompt_ner = ChatPromptTemplate.from_messages([
    ("system", "Extraia do texto os nomes de atores/atrizes e diretores/diretoras de cinema. "
               "Ignore filmes, personagens e qualquer outro nome. Responda sempre em português."),
    ("human", "{texto}"),
])

chain_ner = prompt_ner | llm.with_structured_output(Pessoas)

In [11]:
def filmografia(pessoa):
    """Busca a pessoa no TMDB e devolve seus 5 filmes mais populares (como ator ou como diretor)."""
    achados = tmdb("/search/person", query=pessoa.nome)["results"]
    if not achados:
        return []

    creditos = tmdb(f"/person/{achados[0]['id']}/movie_credits")
    if pessoa.tipo == "diretor":
        filmes = [f for f in creditos["crew"] if f["job"] == "Director"]
    else:
        filmes = creditos["cast"]

    filmes = sorted(filmes, key=lambda f: f["popularity"], reverse=True)[:5]
    return [f"{f['title']} ({f.get('release_date', '')[:4]})" for f in filmes]


textos = [
    "Ontem assisti a um filme dirigido por Denis Villeneuve com a Zendaya e fiquei curioso sobre os outros trabalhos deles.",
    "Quero ver mais coisas do Christopher Nolan e do Cillian Murphy.",
]

for texto in textos:
    print("Texto:", texto)
    for pessoa in chain_ner.invoke({"texto": texto}).pessoas:
        print(f"\n  {pessoa.nome} ({pessoa.tipo}):")
        for filme in filmografia(pessoa):
            print("    -", filme)
    print("\n" + "-" * 60)

Texto: Ontem assisti a um filme dirigido por Denis Villeneuve com a Zendaya e fiquei curioso sobre os outros trabalhos deles.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



  Denis Villeneuve (diretor):
    - Duna: Parte Dois (2024)
    - Duna (2021)
    - Duna: Parte Três (2026)
    - Blade Runner 2049 (2017)
    - A Chegada (2016)

  Zendaya (ator):
    - A Odisseia (2026)
    - Homem-Aranha: Um Novo Dia (2026)
    - Duna: Parte Dois (2024)
    - Duna (2021)
    - Homem-Aranha: Sem Volta Para Casa (2021)

------------------------------------------------------------
Texto: Quero ver mais coisas do Christopher Nolan e do Cillian Murphy.

  Christopher Nolan (diretor):
    - A Odisseia (2026)
    - Interestelar (2014)
    - A Origem (2010)
    - Batman: O Cavaleiro das Trevas (2008)
    - Oppenheimer (2023)

  Cillian Murphy (ator):
    - A Origem (2010)
    - Batman: O Cavaleiro das Trevas (2008)
    - Oppenheimer (2023)
    - Batman: O Cavaleiro das Trevas Ressurge (2012)
    - Batman Begins (2005)

------------------------------------------------------------


# 2. SISTEMA DE PERGUNTAS E RESPOSTAS (RAG)

O TMDB fornece os **dados reais** do filme (elenco, direção, duração, sinopse) e o LLM responde **somente com base neles**.
Essa estratégia se chama **RAG** (Geração Aumentada por Recuperação): primeiro recuperamos informação, depois geramos a resposta.

`pergunta → TMDB (dados do filme) → LLM (responde usando só esses dados)`

In [12]:
# 1) Recupera os dados do filme no TMDB
filme = tmdb("/search/movie", query=FILME)["results"][0]
dados = tmdb(f"/movie/{filme['id']}", append_to_response="credits")

direcao = ", ".join(p["name"] for p in dados["credits"]["crew"] if p["job"] == "Director")
elenco = ", ".join(f"{a['name']} (como {a['character']})" for a in dados["credits"]["cast"][:10])

contexto = f"""Título: {dados['title']}
Lançamento: {dados['release_date']}
Duração: {dados['runtime']} minutos
Gêneros: {", ".join(g['name'] for g in dados['genres'])}
Direção: {direcao}
Elenco principal: {elenco}
Sinopse: {dados['overview']}"""

print(contexto)

Título: Oppenheimer
Lançamento: 2023-07-19
Duração: 181 minutos
Gêneros: Drama, História, Guerra
Direção: Christopher Nolan
Elenco principal: Cillian Murphy (como J. Robert Oppenheimer), Emily Blunt (como Kitty Oppenheimer), Matt Damon (como Leslie Groves), Robert Downey Jr. (como Lewis Strauss), Florence Pugh (como Jean Tatlock), Josh Hartnett (como Ernest Lawrence), Casey Affleck (como Boris Pash), Rami Malek (como David Hill), Kenneth Branagh (como Niels Bohr), Benny Safdie (como Edward Teller)
Sinopse: A história do físico americano J. Robert Oppenheimer, seu papel no Projeto Manhattan e no desenvolvimento da bomba atômica durante a Segunda Guerra Mundial, e o quanto isso mudaria a história do mundo para sempre.


In [13]:
# 2) O LLM responde usando APENAS o contexto
prompt_qa = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de cinema. Responda à pergunta usando SOMENTE as informações do contexto. "
               "Se a resposta não estiver no contexto, diga que não sabe. Responda em português.\n\nContexto:\n{contexto}"),
    ("human", "{pergunta}"),
])

chain_qa = prompt_qa | llm | StrOutputParser()

perguntas = [
    "Quem dirigiu o filme e quem são os atores principais?",
    "Quanto tempo o filme dura?",
    "Do que o filme trata, em uma frase?",
    "Qual foi a bilheteria do filme?",   # não está no contexto: o modelo deve dizer que não sabe
]

for pergunta in perguntas:
    print("Pergunta:", pergunta)
    print("Resposta:", chain_qa.invoke({"contexto": contexto, "pergunta": pergunta}), "\n")

Pergunta: Quem dirigiu o filme e quem são os atores principais?
Resposta: O filme foi dirigido por Christopher Nolan. O elenco principal é composto por:

* Cillian Murphy (como J. Robert Oppenheimer)
* Emily Blunt (como Kitty Oppenheimer)
* Matt Damon (como Leslie Groves)
* Robert Downey Jr. (como Lewis Strauss)
* Florence Pugh (como Jean Tatlock)
* Josh Hartnett (como Ernest Lawrence)
* Casey Affleck (como Boris Pash)
* Rami Malek (como David Hill)
* Kenneth Branagh (como Niels Bohr)
* Benny Safdie (como Edward Teller) 

Pergunta: Quanto tempo o filme dura?
Resposta: O filme dura 181 minutos. 

Pergunta: Do que o filme trata, em uma frase?
Resposta: O filme trata da história do físico americano J. Robert Oppenheimer, seu papel no Projeto Manhattan e no desenvolvimento da bomba atômica durante a Segunda Guerra Mundial, e o quanto isso mudaria a história do mundo para sempre. 

Pergunta: Qual foi a bilheteria do filme?
Resposta: Eu não sei. 



# 3. CLASSIFICAÇÃO DE TEXTOS

O usuário descreve **o que quer assistir**. O LLM **classifica o pedido em gêneros** (e detecta a emoção desejada), o TMDB devolve filmes reais desses gêneros e o LLM explica a recomendação.

`pedido → LLM (classifica em gêneros) → TMDB (descobre filmes) → LLM (explica por que combinam)`

In [14]:
class Pedido(BaseModel):
    generos: List[str] = Field(description="1 ou 2 gêneros da lista que melhor combinam com o pedido")
    emocao: str = Field(description="Emoção que o usuário quer sentir, em uma palavra (ex.: medo, alegria, tensão)")

prompt_pedido = ChatPromptTemplate.from_messages([
    ("system", "Classifique o pedido do usuário em 1 ou 2 gêneros de filme. "
               "Escolha apenas gêneros desta lista, escritos exatamente como estão: " + ", ".join(mapa_generos) + ". "
               "Responda sempre em português."),
    ("human", "{pedido}"),
])

prompt_recomendacao = ChatPromptTemplate.from_messages([
    ("system", "Você recomenda filmes. Para cada filme da lista, escreva UMA frase em português "
               "explicando por que combina com o pedido do usuário.\n\nPedido: {pedido}"),
    ("human", "{filmes}"),
])

chain_pedido = prompt_pedido | llm.with_structured_output(Pedido)
chain_recomendacao = prompt_recomendacao | llm | StrOutputParser()

In [15]:
def recomendar(pedido):
    print("Pedido:", pedido)

    # 1) LLM classifica o pedido
    classificacao = chain_pedido.invoke({"pedido": pedido})
    print("Gêneros:", ", ".join(classificacao.generos), "| Emoção:", classificacao.emocao)

    # 2) TMDB busca filmes desses gêneros (os mais populares)
    ids = [str(mapa_generos[g.lower()]) for g in classificacao.generos if g.lower() in mapa_generos]
    filmes = tmdb("/discover/movie", with_genres=",".join(ids), sort_by="popularity.desc")["results"][:5]
    lista = "\n".join(f"- {f['title']}: {f['overview'][:200]}" for f in filmes)

    # 3) LLM explica por que cada filme combina com o pedido
    print("\n" + chain_recomendacao.invoke({"pedido": pedido, "filmes": lista}))
    print("\n" + "-" * 60)


recomendar("Quero algo tenso e sombrio para ver de madrugada.")
recomendar("Tive um dia difícil e preciso rir muito.")

Pedido: Quero algo tenso e sombrio para ver de madrugada.
Gêneros: terror, thriller | Emoção: tensão

Aqui estão as recomendações para a sua madrugada:

- **Apocalipse Z: Novo Mundo**: Perfeito para a madrugada por mergulhar em uma atmosfera desoladora de sobrevivência extrema e isolamento aterrorizante.
- **Boiúna - Terror no Amazonas**: Traz o suspense claustrofóbico ideal para assistir no escuro, misturando ameaças humanas e o desconhecido na selva.
- **Obsessão**: Uma escolha perturbadora que transforma o que seria um romance em um pesadelo psicológico imprevisível.
- **Sobrenatural: Agora Entre Nós**: Garante sustos e uma carga pesada de terror sobrenatural para quem quer ver o terror invadir a realidade nas horas tardias.
- **Cara-de-Barro**: Mergulha nas sombras e na tragédia sombria de Gotham, entregando um tom niilista e tenso do início ao fim.

------------------------------------------------------------
Pedido: Tive um dia difícil e preciso rir muito.
Gêneros: comédia | Emoç

# 4. ANÁLISE DE SENTIMENTOS + TRADUÇÃO + SUMARIZAÇÃO

O TMDB fornece avaliações de espectadores (em **inglês**). O LLM, para cada uma, **classifica o sentimento**, **traduz e resume** para o português. No fim, gera um **veredito geral** do filme.

`avaliações (TMDB, em inglês) → LLM (sentimento + tradução + resumo) → LLM (veredito geral)`

In [16]:
# 1) Busca as avaliações do filme no TMDB (em inglês)
filme = tmdb("/search/movie", query=FILME)["results"][0]
avaliacoes = tmdb(f"/movie/{filme['id']}/reviews", language="en-US")["results"][:5]

print(f"Filme: {filme['title']} | {len(avaliacoes)} avaliações")
print("\nExemplo de avaliação original:\n", avaliacoes[0]["content"][:300], "...")

Filme: Oppenheimer | 5 avaliações

Exemplo de avaliação original:
 FULL SPOILER-FREE REVIEW @ https://www.firstshowing.net/2023/review-nolans-oppenheimer-is-a-harrowing-tale-of-one-mans-life/

"Oppenheimer is a true masterclass in how to build extreme tension and suspense through fast, detailed dialogue, an insanely powerful sound production, and an equally explo ...


In [17]:
# 2) Para cada avaliação: sentimento + resumo traduzido para o português
class Avaliacao(BaseModel):
    sentimento: Literal["positivo", "negativo", "neutro"] = Field(description="Opinião do espectador sobre o filme")
    resumo: str = Field(description="Resumo da avaliação em UMA frase, traduzido para o português")

prompt_avaliacao = ChatPromptTemplate.from_messages([
    ("system", "Você analisa avaliações de filmes escritas em inglês. Classifique o sentimento do espectador "
               "e resuma a avaliação em uma frase, traduzida para o português."),
    ("human", "{avaliacao}"),
])

chain_avaliacao = prompt_avaliacao | llm.with_structured_output(Avaliacao)

entradas = [{"avaliacao": a["content"][:1500]} for a in avaliacoes]
resultados = chain_avaliacao.batch(entradas)

df = pd.DataFrame({
    "nota": [a["author_details"]["rating"] for a in avaliacoes],
    "sentimento": [r.sentimento for r in resultados],
    "resumo (em português)": [r.resumo for r in resultados],
})
df

,nota,sentimento,resumo (em português)
0,8.0,positivo,"Oppenheimer é uma verdadeira aula de cinema, oferecendo uma experiência tensa, perturbadora e magistralmente atuada, embora seu ritmo pesado possa não agradar a todos."
1,10.0,positivo,"O filme é uma obra-prima cinematográfica e biográfica, ideal para quem aprecia a história da ciência, destacando-se pelas atuações brilhantes, trilha sonora impecável e visuais impressionantes."
2,9.0,positivo,Oppenheimer é uma obra-prima cinematográfica impressionante e a melhor obra da carreira de Christopher Nolan.
3,7.0,positivo,"O filme apresenta excelentes atuações de Cillian Murphy e Robert Downey Jr., focando no desenvolvimento histórico do Projeto Manhattan."
4,NaN,positivo,"O filme oferece uma jornada histórica emocionante, combinando uma narrativa cativante, visuais deslumbrantes e atuações de alto nível."


In [18]:
# 3) Veredito geral a partir dos resumos
prompt_veredito = ChatPromptTemplate.from_messages([
    ("system", "Com base nos resumos de avaliações do filme {filme}, escreva em português um veredito geral "
               "em 3 frases: o que os espectadores elogiaram, o que criticaram e se a recepção foi positiva, negativa ou mista."),
    ("human", "{resumos}"),
])

chain_veredito = prompt_veredito | llm | StrOutputParser()

resumos = "\n".join(f"- ({r.sentimento}) {r.resumo}" for r in resultados)
print(chain_veredito.invoke({"filme": filme["title"], "resumos": resumos}))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Os espectadores elogiaram amplamente o filme pelas atuações brilhantes de Cillian Murphy e Robert Downey Jr., pela direção magistral de Christopher Nolan, além da trilha sonora impecável e dos visuais impressionantes. Como principal ponto negativo, apontou-se que o ritmo pesado da narrativa pode não agradar a todos os públicos. No geral, a recepção foi extremamente positiva, sendo aclamado como uma obra-prima cinematográfica e biográfica.


# CONCLUSÃO

| Técnica | O que o usuário fornece | O que o TMDB traz | Resultado do LLM |
|---|---|---|---|
| NER | Texto livre citando pessoas | Filmografia (créditos) | Atores e diretores identificados |
| Perguntas e Respostas (RAG) | Perguntas sobre um filme | Elenco, direção, duração, sinopse | Respostas baseadas nos dados reais |
| Classificação de textos | Pedido do que quer assistir | Filmes dos gêneros escolhidos | Gêneros, emoção e recomendações |
| Sentimentos + Tradução + Sumarização | (nenhum) | Avaliações em inglês | Sentimento, resumo em português e veredito |

Em todas, o **LangChain** liga o LLM à API em *chains* simples, e o **TMDB** fornece dados reais que o modelo não precisa inventar.